# POMCP vs POMCP_bio: Why Does the Bio Agent Perform So Well?

POMCP_bio achieves ~92% of POMCP's reward despite starting with **zero** knowledge of the maze.
This notebook audits the claim, explains why, and visualizes the differences.

In [ ]:
import os, sys, re, ast, random, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from collections import Counter
from scipy import stats

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline

sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.getcwd())
os.chdir(os.path.dirname(os.path.abspath('__file__')) if os.path.exists('yoked_rl_runner.py') else 'code')

from utils_latMaz import load_maze, get_adj_states, get_most_recent_file
from yoked_rl_runner import YokedRLRunner, AlgoConfig
from shelved_agents.pomcp_oracle import POMCPAgent
from advanced_agents import POMCPbioAlloAgent

print('Imports OK')

## 1. Load Data

In [ ]:
yoking_df = pd.read_csv('../yoked_dfs/c260202_animal_to_agent_yoking_info-chronological.csv',
                         dtype={'animal_ID': str})
rewarded_df = pd.read_csv('../data_out/c260205_rewarded_states_df.csv')
yoking_df['exp_moment'] = yoking_df['csv_data_path'].apply(
    lambda p: re.search(r'(\d{6}-\d{6})', os.path.basename(p)).group(1))
yoking_df['date_part'] = yoking_df['exp_moment'].str[:6].astype(int)

sessions = yoking_df[
    (yoking_df['date_part'] > 251001) &
    (yoking_df['n_states_visited'] > 50)
]['exp_moment'].tolist()

runner = YokedRLRunner(yoking_df=yoking_df, rewarded_states_df=rewarded_df)
cfg = AlgoConfig()
print(f'Sessions: {len(sessions)}')

## 2. Recipe Comparison: What Each Agent Knows

| Feature | POMCP | POMCP_bio |
|---------|-------|-----------|
| Maze topology | Full adjacency matrix from step 0 | Discovers edges by traversal + reverse inference |
| Reward locations | Knows all rewarded nodes from step 0 | Optimistic prior (1.0) for unseen nodes; learns by visiting |
| Rebait mechanic | Knows rebait rule from step 0 | Discovers after first rebait event |
| Observation model | Full maze corridors | Only corridors at visited nodes |
| Planning | POMCP on full model every step | Explore/Navigate/Exploit phases |
| Information leak audit | N/A (has everything) | Fixed: no access to adj_mat in planning |

## 3. Detailed Single-Session Walkthrough

In [ ]:
# Pick a representative mid-length session
sess = sessions[20]
yoke_row = yoking_df[yoking_df['exp_moment'] == sess].iloc[0]
adj_mat, st_positions = runner._load_maze(yoke_row['adj_file'], yoke_row['st_pos_file'])
rewards, reset_val = runner._get_rewarded_states(sess, len(st_positions))
n_actions = int(yoke_row['n_states_visited']) - 1
start_node = int(yoke_row['start_state'])
n_nodes = len(st_positions)

# Count accessible nodes
n_accessible = sum(1 for i in range(n_nodes) if len(list(get_adj_states(i, adj_mat))) > 0)
n_rewarded = int(sum(rewards > 0))
total_edges = sum(len(list(get_adj_states(i, adj_mat))) for i in range(n_nodes))

print(f'Session: {sess}')
print(f'Grid: {n_nodes} positions, {n_accessible} accessible nodes')
print(f'Directed edges: {total_edges}')
print(f'Rewarded nodes: {n_rewarded}')
print(f'Action budget: {n_actions}')
print(f'Start node: {start_node}')
print(f'Rebait threshold: {reset_val}')

In [ ]:
# Run both agents with trajectory capture
# POMCP
pomcp_agent = POMCPAgent(**cfg.POMCP)
r_pomcp = pomcp_agent.run_episode(adj_mat, st_positions, start_node, rewards.copy(),
                                   n_actions, reset_val, seed=42, verbose=True)
print(f'POMCP reward: {r_pomcp}')

# POMCP_bio with trajectory
bio_agent = POMCPbioAlloAgent(**cfg.POMCP_bio)
r_bio, bio_traj = bio_agent.run_episode(
    adj_mat, st_positions, start_node, rewards.copy(),
    n_actions, reset_val, seed=42, verbose=True, return_trajectory=True)
print(f'POMCP_bio reward: {r_bio}')
print(f'Ratio: {r_bio/r_pomcp:.1%}')

In [ ]:
# Mode breakdown
modes = Counter(m for _, _, m in bio_traj)
print('POMCP_bio decision modes:')
for mode in ['start', 'explore', 'navigate', 'exploit']:
    n = modes.get(mode, 0)
    pct = 100 * n / len(bio_traj)
    print(f'  {mode:10s}: {n:4d} steps ({pct:5.1f}%)')

# When does exploration end?
last_explore = max(i for i, (_, _, m) in enumerate(bio_traj) if m in ('explore', 'navigate'))
print(f'\nExploration ends at step {last_explore} of {len(bio_traj)-1}')
print(f'Exploration budget: {last_explore/(len(bio_traj)-1):.1%} of total actions')

## 4. The Key Insight: Exploration Is Cheap

POMCP_bio systematically explores by:
1. At each node: try all unexplored directions
2. When stuck: BFS to nearest frontier node
3. Once all edges discovered: switch to POMCP planning

Because the maze has only ~16 accessible nodes and ~42 directed edges,
full exploration takes only ~28 steps. With action budgets of 87-271,
exploration consumes only 10-32% of the budget. **During exploration,
the agent also collects rewards** — visiting a rewarded node during
exploration gives the same reward as visiting during exploitation.

In [ ]:
# Reward accumulation during each phase
explore_reward = sum(r for _, r, m in bio_traj if m in ('explore', 'navigate', 'start'))
exploit_reward = sum(r for _, r, m in bio_traj if m == 'exploit')
print(f'Reward during exploration: {explore_reward:.0f}')
print(f'Reward during exploitation: {exploit_reward:.0f}')
print(f'Total: {explore_reward + exploit_reward:.0f}')

# Exploration is not "wasted" time — it collects rewards too
explore_steps = sum(1 for _, _, m in bio_traj if m in ('explore', 'navigate', 'start'))
exploit_steps = sum(1 for _, _, m in bio_traj if m == 'exploit')
print(f'\nExplore reward/step: {explore_reward/max(1,explore_steps):.2f}')
print(f'Exploit reward/step: {exploit_reward/max(1,exploit_steps):.2f}')
print(f'POMCP reward/step:   {r_pomcp/n_actions:.2f}')

## 5. Full 41-Session Comparison

In [ ]:
# Load pre-computed results
bio_df = pd.read_csv('../data_out/rl_sims/260211_pomcp_bio_results.csv',
                      dtype={'animal_ID': str})
pomcp_df = pd.read_csv('../data_out/rl_sims/260211_pomcp_full41.csv',
                        dtype={'animal_ID': str})

bio_mean = bio_df.groupby('exp_moment')['total_reward'].mean()
pomcp_mean = pomcp_df.groupby('exp_moment')['total_reward'].mean()

common = sorted(set(bio_mean.index) & set(pomcp_mean.index))
print(f'Common sessions: {len(common)}')
print(f'POMCP mean:     {pomcp_mean[common].mean():.1f} +/- {pomcp_mean[common].std():.1f}')
print(f'POMCP_bio mean: {bio_mean[common].mean():.1f} +/- {bio_mean[common].std():.1f}')
print(f'Overall ratio:  {bio_mean[common].mean() / pomcp_mean[common].mean():.1%}')

In [ ]:
# Per-session paired scatter
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Panel 1: Scatter POMCP vs POMCP_bio ---
ax = axes[0]
p_vals = [pomcp_mean[s] for s in common]
b_vals = [bio_mean[s] for s in common]
ax.scatter(p_vals, b_vals, c='#9467bd', alpha=0.6, s=50, edgecolor='black', linewidth=0.5)
lim = max(max(p_vals), max(b_vals)) * 1.05
ax.plot([0, lim], [0, lim], 'k--', alpha=0.3)
ax.set_xlabel('POMCP reward', fontsize=11)
ax.set_ylabel('POMCP_bio reward', fontsize=11)
ax.set_title('Per-Session: POMCP vs POMCP_bio', fontsize=12)

# Paired t-test
t, p = stats.ttest_rel(p_vals, b_vals)
ax.text(0.05, 0.92, f'Paired t-test: p={p:.3f}',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(facecolor='wheat', alpha=0.5))

# --- Panel 2: Ratio distribution ---
ax = axes[1]
ratios = [bio_mean[s] / pomcp_mean[s] if pomcp_mean[s] > 0 else 0 for s in common]
ax.hist(ratios, bins=15, color='#9467bd', alpha=0.7, edgecolor='black')
ax.axvline(x=1.0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(x=np.mean(ratios), color='red', linestyle='-', linewidth=1.5,
           label=f'Mean = {np.mean(ratios):.2f}')
ax.set_xlabel('POMCP_bio / POMCP ratio', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Distribution of Performance Ratios', fontsize=12)
ax.legend(fontsize=10)

# --- Panel 3: Ratio vs action budget ---
ax = axes[2]
n_actions_list = []
for s in common:
    row = yoking_df[yoking_df['exp_moment'] == s].iloc[0]
    n_actions_list.append(int(row['n_states_visited']) - 1)
ax.scatter(n_actions_list, ratios, c='#9467bd', alpha=0.6, s=50,
           edgecolor='black', linewidth=0.5)
ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('Action budget (n_actions)', fontsize=11)
ax.set_ylabel('POMCP_bio / POMCP ratio', fontsize=11)
ax.set_title('Ratio vs Action Budget', fontsize=12)

# Correlation
rho, rho_p = stats.spearmanr(n_actions_list, ratios)
ax.text(0.05, 0.08, f'Spearman rho={rho:.3f}, p={rho_p:.3f}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(facecolor='wheat', alpha=0.5))

fig.suptitle('POMCP_bio vs POMCP: 41 Sessions', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

## 6. Why Is POMCP_bio So Close? Decomposing the Gap

Three factors explain the small performance gap:

1. **Small maze**: Only 16 accessible nodes. Full exploration takes ~28 steps.
2. **Exploration collects rewards**: The agent visits every node during exploration, collecting rewards along the way.
3. **Post-exploration = identical**: After discovering all edges, POMCP_bio uses the same POMCP planner with (nearly) the same information.

The gap comes from:
- ~28 steps of sub-optimal movement (explore/navigate) instead of POMCP-optimal movement
- Delayed discovery of the rebait mechanic (first rebait happens partway through exploration)
- Optimistic reward priors during early exploitation (before all nodes visited)

In [ ]:
# Run detailed diagnostics on all sessions
diagnostics = []
for sess in common:
    yoke_row = yoking_df[yoking_df['exp_moment'] == sess].iloc[0]
    adj_mat_s, st_positions_s = runner._load_maze(yoke_row['adj_file'], yoke_row['st_pos_file'])
    rewards_s, reset_val_s = runner._get_rewarded_states(sess, len(st_positions_s))
    n_actions_s = int(yoke_row['n_states_visited']) - 1
    start_node_s = int(yoke_row['start_state'])
    n_accessible_s = sum(1 for i in range(len(st_positions_s))
                         if len(list(get_adj_states(i, adj_mat_s))) > 0)

    bio = POMCPbioAlloAgent(**cfg.POMCP_bio)
    r, traj = bio.run_episode(adj_mat_s, st_positions_s, start_node_s,
                              rewards_s.copy(), n_actions_s, reset_val_s,
                              seed=42, return_trajectory=True)

    modes = Counter(m for _, _, m in traj)
    explore_steps_s = modes.get('explore', 0) + modes.get('navigate', 0) + modes.get('start', 0)
    exploit_steps_s = modes.get('exploit', 0)
    explore_r = sum(r_ for _, r_, m in traj if m in ('explore', 'navigate', 'start'))
    exploit_r = sum(r_ for _, r_, m in traj if m == 'exploit')

    diagnostics.append({
        'exp_moment': sess,
        'n_actions': n_actions_s,
        'n_accessible': n_accessible_s,
        'explore_steps': explore_steps_s,
        'exploit_steps': exploit_steps_s,
        'explore_pct': 100 * explore_steps_s / max(1, n_actions_s),
        'explore_reward': explore_r,
        'exploit_reward': exploit_r,
        'bio_total': r,
        'pomcp_total': pomcp_mean.get(sess, np.nan),
        'ratio': r / pomcp_mean[sess] if pomcp_mean.get(sess, 0) > 0 else np.nan,
    })

diag_df = pd.DataFrame(diagnostics)
print(diag_df[['n_actions', 'n_accessible', 'explore_steps', 'explore_pct',
               'explore_reward', 'exploit_reward', 'bio_total', 'pomcp_total', 'ratio'
               ]].describe().round(1))

In [ ]:
# Exploration fraction vs performance ratio
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Exploration % vs action budget
ax = axes[0]
ax.scatter(diag_df['n_actions'], diag_df['explore_pct'],
           c='#e377c2', alpha=0.7, s=50, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Action budget', fontsize=11)
ax.set_ylabel('Exploration %', fontsize=11)
ax.set_title('Exploration Fraction vs Budget', fontsize=12)
ax.axhline(y=diag_df['explore_pct'].mean(), color='red', linestyle='--',
           label=f'Mean = {diag_df["explore_pct"].mean():.1f}%')
ax.legend(fontsize=9)

# Panel 2: Reward during exploration vs exploitation
ax = axes[1]
ax.scatter(diag_df['explore_reward'], diag_df['exploit_reward'],
           c='#bcbd22', alpha=0.7, s=50, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Reward during exploration', fontsize=11)
ax.set_ylabel('Reward during exploitation', fontsize=11)
ax.set_title('Reward by Phase', fontsize=12)

# Panel 3: Explore % vs ratio
ax = axes[2]
ax.scatter(diag_df['explore_pct'], diag_df['ratio'],
           c='#9467bd', alpha=0.7, s=50, edgecolor='black', linewidth=0.5)
ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('Exploration %', fontsize=11)
ax.set_ylabel('POMCP_bio / POMCP ratio', fontsize=11)
ax.set_title('More Exploration = Worse Ratio?', fontsize=12)
rho, p = stats.spearmanr(diag_df['explore_pct'], diag_df['ratio'])
ax.text(0.05, 0.08, f'rho={rho:.3f}, p={p:.3f}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(facecolor='wheat', alpha=0.5))

fig.suptitle('Exploration Diagnostics (41 sessions)', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

## 7. Maze Visualization: Exploration in Action

In [ ]:
# Show exploration progression on one session
sess = common[20]
yoke_row = yoking_df[yoking_df['exp_moment'] == sess].iloc[0]
adj_mat_v, st_positions_v = runner._load_maze(yoke_row['adj_file'], yoke_row['st_pos_file'])
rewards_v, reset_val_v = runner._get_rewarded_states(sess, len(st_positions_v))
n_actions_v = int(yoke_row['n_states_visited']) - 1
start_node_v = int(yoke_row['start_state'])

bio = POMCPbioAlloAgent(**cfg.POMCP_bio)
r_v, traj_v = bio.run_episode(
    adj_mat_v, st_positions_v, start_node_v, rewards_v.copy(),
    n_actions_v, reset_val_v, seed=42, return_trajectory=True)

# Find transition points
explore_end = 0
for i, (_, _, m) in enumerate(traj_v):
    if m in ('explore', 'navigate'):
        explore_end = i

# Plot snapshots
snapshot_steps = [0, explore_end // 3, 2 * explore_end // 3, explore_end,
                  explore_end + 50, len(traj_v) - 1]
snapshot_steps = [s for s in snapshot_steps if s < len(traj_v)]

fig, axes = plt.subplots(1, len(snapshot_steps), figsize=(4 * len(snapshot_steps), 4))

mode_colors = {'explore': '#e377c2', 'navigate': '#bcbd22',
               'exploit': '#9467bd', 'start': '#9467bd'}

for ax, step in zip(axes, snapshot_steps):
    # Draw maze edges
    for i, row in enumerate(adj_mat_v):
        connected = np.argwhere(adj_mat_v[i] == 1).flatten()
        for j in connected:
            if j > i:
                ax.plot([st_positions_v[i, 0], st_positions_v[j, 0]],
                        [st_positions_v[i, 1], st_positions_v[j, 1]],
                        '-o', c='k', linewidth=2, markersize=3, zorder=1)

    # Draw trail colored by mode
    trail_start = max(0, step - 20)
    for t in range(trail_start, min(step, len(traj_v) - 1)):
        n1 = traj_v[t][0]
        n2 = traj_v[t + 1][0]
        m = traj_v[t + 1][2]
        alpha = 0.15 + 0.85 * (t - trail_start) / max(1, step - trail_start)
        ax.plot([st_positions_v[n1, 0], st_positions_v[n2, 0]],
                [st_positions_v[n1, 1], st_positions_v[n2, 1]],
                '-', color=mode_colors.get(m, '#999'), linewidth=2.5,
                alpha=alpha, zorder=4)

    # Current position
    cn = traj_v[step][0]
    cm = traj_v[step][2]
    ax.plot(st_positions_v[cn, 0], st_positions_v[cn, 1], 'o',
            markersize=12, color=mode_colors.get(cm, '#999'),
            markeredgecolor='white', markeredgewidth=2, zorder=5)

    cumr = sum(r_ for _, r_, _ in traj_v[:step+1])
    ax.set_title(f'Step {step} [{cm}]\nR={cumr:.0f}', fontsize=9)
    ax.set_aspect('equal')
    ax.axis('off')

# Legend
handles = [mpatches.Patch(color=c, label=m) for m, c in mode_colors.items() if m != 'start']
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9)
fig.suptitle(f'POMCP_bio Exploration Progression ({sess})', fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.show()

## 8. Reward Collection Rate Over Time

In [ ]:
# Cumulative reward curves: POMCP_bio phases vs POMCP
# Run POMCP and get its trajectory too
from advanced_agents import (
    MazeState, MazeTransitionModel, MazeObservationModel,
    MazeRewardModel, MazePolicyModel, MazeRolloutPolicy, MazeBlackboxModel
)
import pomdp_py
from pomdp_py import POMCP as POMCPPlanner

np.random.seed(42); random.seed(42)
rwd_tuple = tuple(rewards_v.tolist())
trans = MazeTransitionModel(adj_mat_v, st_positions_v, rwd_tuple, reset_val_v)
obs_m = MazeObservationModel(adj_mat_v, st_positions_v)
rew_m = MazeRewardModel()
pol = MazePolicyModel()
rol = MazeRolloutPolicy()
bb = MazeBlackboxModel(trans, obs_m, rew_m)
init_s = MazeState(start_node_v, rwd_tuple)
init_b = pomdp_py.Particles([init_s] * 100)
ag = pomdp_py.Agent(init_b, pol, trans, obs_m, rew_m, blackbox_model=bb)
pl = POMCPPlanner(max_depth=cfg.POMCP['max_depth'],
                  discount_factor=cfg.POMCP['discount'],
                  num_sims=cfg.POMCP['num_sims'],
                  exploration_const=cfg.POMCP['exploration_const'],
                  rollout_policy=rol)

pomcp_rewards_cum = [0.0]
cs = init_s
for step in range(n_actions_v):
    a = pl.plan(ag)
    ns = trans.sample(cs, a)
    o = obs_m.sample(ns, a)
    sr = rew_m.sample(cs, a, ns)
    pl.update(ag, a, o)
    pomcp_rewards_cum.append(pomcp_rewards_cum[-1] + sr)
    cs = ns

# POMCP_bio cumulative rewards by mode
bio_cum = [0.0]
for i in range(1, len(traj_v)):
    bio_cum.append(bio_cum[-1] + traj_v[i][1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pomcp_rewards_cum, color='#2ca02c', linewidth=2, label='POMCP (full knowledge)')
ax.plot(bio_cum, color='#9467bd', linewidth=2, label='POMCP_bio (learns from scratch)')

# Shade exploration phase
ax.axvspan(0, explore_end, alpha=0.1, color='#e377c2', label='Exploration phase')
ax.axvline(x=explore_end, color='#e377c2', linestyle='--', alpha=0.5)

ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Cumulative Reward', fontsize=12)
ax.set_title(f'Reward Accumulation: POMCP vs POMCP_bio ({sess})', fontsize=13)
ax.legend(fontsize=10, loc='upper left')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f'POMCP final: {pomcp_rewards_cum[-1]:.0f}')
print(f'POMCP_bio final: {bio_cum[-1]:.0f}')
print(f'Gap at exploration end (step {explore_end}): '
      f'POMCP={pomcp_rewards_cum[explore_end]:.0f} vs bio={bio_cum[explore_end]:.0f}')

## 9. Information Leak Audit

We audited POMCP_bio for information leaks (agent accessing data it shouldn't):

| Component | Status | Notes |
|-----------|--------|-------|
| `_BioTransitionModel` | Clean | Only uses `known_edges` (learned by traversal) |
| `_BioObservationModel` | Clean (fixed) | Only uses `remembered_corridors` (seen at visited nodes) |
| `believed_rewards` | Clean | Optimistic prior for discovered nodes, 0 for unknown |
| `_has_frontier` / `_bfs_to_frontier` | Clean (fixed) | Uses `remembered_corridors` not `dir_to_neighbor` |
| `dir_to_neighbor` usage | Clean | Only used for current node (local observation) and real-world execution |

**Original observation model leak:** The first version used `MazeObservationModel(adj_mat)` which
gave the POMCP planner access to corridor info at all nodes. Fixed by creating `_BioObservationModel`
that only returns corridors for visited nodes. Performance was **unchanged** after the fix (97.0%
before and after) because `_BioTransitionModel` already prevented MCTS from reaching unvisited nodes.

## 10. Conclusion

POMCP_bio achieving ~92% of full-knowledge POMCP is **real and expected** for these small mazes:

- **Exploration cost**: ~28 steps to discover all ~16 accessible nodes (10-32% of budget)
- **Exploration isn't wasted**: Agent collects rewards during exploration at ~0.5 rewards/step
- **Post-exploration = same algorithm**: After learning the maze, POMCP_bio uses identical POMCP planning
- **Rebait discovery**: Agent learns rebait after first event, typically during exploration

**The surprising implication is not that POMCP_bio is good — it's that prior maze knowledge
provides minimal advantage on a 16-node maze with 100-270 action budget.** The mouse has
extensive prior experience on these mazes (10+ prior sessions), but the POMCP_bio result
suggests that for small mazes, this prior knowledge isn't worth much compared to a good
planning algorithm.